In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
import h5py

import openquantum_sde
from openquantum_sde.io import load_trajectory, load_params
from openquantum_sde.utils import calculate_num_photons, calculate_num_photons_chunk


In [ ]:
# Lookup data folder
if "DATA" in os.environ:
    base_dir = Path(os.environ["DATA"]).expanduser()
else:
    base_dir = Path(".")  # current folder
PROJECT_NAME = "openquantum_sde"

In [ ]:
# Function to calculate number of photons
def calculate_num_photons_from_trajfiles(output_dir, chunk_size, nreps=1):
    '''
    Loads a random continuous chunk of trajectory nreps and calculate the average number of photons
    '''
    file_list = [
        Path(output_dir) / f"traj_CK_{i:04d}.h5"
        for i in range(1, 11)
    ]

    total_photons = 0.0
    total_norm = 0.0

    for _ in range(nreps):

        filename = np.random.choice(file_list)

        with h5py.File(filename, "r") as f:
            traj = f["traj"]

            start = np.random.randint(
                0, traj.shape[0] - chunk_size
            )

            chunk = traj[start:start + chunk_size]

            photons, norm = calculate_num_photons_chunk(chunk)

            total_photons += photons
            total_norm += norm

    return total_photons / total_norm

In [ ]:
# Load data
chunk_size = 10000; nreps = 50
#chunk_size = 5; nreps = 1
#epsilon_array = list(range(1, 6)) + [x / 4 for x in range(21, 101)] + list(range(26, 31))
epsilon_array = list(range(2, 30 + 1)) #[10]
num_photons = [None] * len(epsilon_array)
for i, epsilon in enumerate(epsilon_array):
    SIM_NAME = "transmon_cavity_eps_" + str(epsilon) + "/data"
    simulation_dir = base_dir / PROJECT_NAME / SIM_NAME
    if not os.path.exists(simulation_dir):
        SIM_NAME = "transmon_cavity_eps_" + str(int(epsilon)) + "/data"
        simulation_dir = base_dir / PROJECT_NAME / SIM_NAME
    num_photons[i] = calculate_num_photons_from_trajfiles(simulation_dir, chunk_size, nreps)
    print(f'\rCalculation eps={epsilon} done      ', end='')

In [ ]:
# Choose values to actually plot
#epsilon_array_plot = np.array(list(range(1, 30 + 1)))
#epsilon_array_plot = [x / 2 for x in range(10, 51)]
#num_photons_plot = np.array(num_photons)[np.isin(np.array(epsilon_array), epsilon_array_plot)]
epsilon_array_plot = np.array(epsilon_array)
num_photons_plot = np.array(num_photons)
print(epsilon_array_plot, num_photons_plot)

In [ ]:
print(epsilon_array)
print(num_photons)

print(epsilon_array_plot)
print(num_photons_plot)
# Plot setup
save_fig = True
fig, ax = plt.subplots(figsize=(5.5, 3.3), dpi=120)
# Set Background Colors
ax.set_facecolor('gainsboro')         # Inner plot area background
# Configure White Grid Lines
ax.grid(True, color='white', linestyle='-', linewidth=1.2)
# Improve visibility of ticks and labels against light gray
ax.tick_params(colors='black')
for spine in ax.spines.values():
    spine.set_edgecolor('black')
# Set global font to sans-serif (Helvetica/Arial)
plt.rcParams.update({
    "font.family": "STIXGeneral",
    "mathtext.fontset": "stix",
    "font.size": 14,
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 14,
})

# Plot
ax.plot(epsilon_array_plot, num_photons_plot, ls='-',  marker='x', ms=5.0, markeredgecolor='darkred',
         label=r'$\langle n_{ph} \rangle$',  lw=2.5, color='black', alpha = 1.0)
ax.set_xlabel(r'drive $(\epsilon)$')
ax.set_ylabel(r'photon number')
#ax.set_xlim([10,15])
#ax.set_ylim([-1,20])
ax.legend(loc='upper left', 
              framealpha=1, 
              fancybox=True,
              edgecolor='0.5',
              borderpad=0.3,
              handlelength=1.2,
              handletextpad=0.4,
              labelspacing=0.3)
ax.grid(True, alpha=0.3)
if save_fig:
    fname = "transmon_cavity_fulleps_vs_nphotons_chunksize" + str(chunk_size) + "_nreps" + str(nreps) + ".png"
    output_figs_dir = base_dir / PROJECT_NAME / "figs"
    output_figs_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_figs_dir / fname)

In [ ]:
# Check size and shape of h5 file
SIM_NAME = "transmon_cavity_eps_" + str(10.0) + "/data"
simulation_dir = base_dir / PROJECT_NAME / SIM_NAME
fname = "traj_CK_0001.h5"
filename = simulation_dir / fname
with h5py.File(filename, "r") as f:
    traj = f["traj"]
    print("shape:", traj.shape)
    print("chunks:", traj.chunks)
    print("size GB:", traj.size * traj.dtype.itemsize / 1e9)